# Part 1.3: Class Imbalance

In [35]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler


In [41]:
# Load the datasets
train=pd.read_csv("C:\\Users\\Hzaab\\Desktop\\MLSD project\\data\\raw\\train.csv")

train=train.drop(columns=["Unnamed: 0"])

log_cols = ["#follows", "#followers", "description length", "#posts"]
for col in log_cols:
    train[col] = np.log1p(train[col])

categorical_cols = ["profile pic", "name==username", "external URL", "private"]
numeric_columns = [col for col in train.columns if col not in categorical_cols and col != "fake"]

In [42]:
X=train.drop(columns=["fake"])
y=train["fake"]
X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=10,
    )


## Logistic regression Baseline

In [43]:
# Preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)



In [44]:
# Class imbalance strategies
strategies = {
    "none": None,
    "class_weight_balanced": "balanced",
    "random_oversample": RandomOverSampler(random_state=10),
    "smote": SMOTE(random_state=10),
    "random_undersample": RandomUnderSampler(random_state=10),
}

In [ ]:
# Experiment tracking
mlflow.set_experiment("class-imbalance-comparison")

# Fix: Remove 'fake' from categorical_cols as it is the target variable, not a feature
categorical_cols = ['profile pic', 'name==username', 'external URL', 'private']

results = []

for strategy_name, strategy in strategies.items():
    run_name = f"logreg_{strategy_name}"

    with mlflow.start_run(run_name=run_name):
        # Build model/pipeline
        if strategy_name == "class_weight_balanced":
            model = LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=10,
            )

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model),
                ]
            )
        elif strategy is None:
            model = LogisticRegression(
                max_iter=2000,
                random_state=10,
            )

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", model),
                ]
            )
        else:
            model = LogisticRegression(
                max_iter=2000,
                random_state=10,
            )

            pipeline = ImbPipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("sampler", strategy),
                    ("model", model),
                ]
            )

        # Fit
        pipeline.fit(X_train, y_train)

        # Predict
        y_pred = pipeline.predict(X_val)
        y_prob = None
        if hasattr(pipeline, "predict_proba"):
            y_prob = pipeline.predict_proba(X_val)[:, 1]

        # Metrics
        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)

        report = classification_report(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)

        # Log params
        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("imbalance_strategy", strategy_name)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 10)
        mlflow.log_param("n_numeric_cols", len(numeric_columns))
        mlflow.log_param("n_categorical_cols", len(categorical_cols))

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        # Save artifacts
        report_path = f"classification_report_{strategy_name}.txt"
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report)

        cm_path = f"confusion_matrix_{strategy_name}.txt"
        with open(cm_path, "w", encoding="utf-8") as f:
            f.write(np.array2string(cm))

        mlflow.log_artifact(report_path)
        mlflow.log_artifact(cm_path)

        # Log model
        mlflow.sklearn.log_model(pipeline, name="model")

        # Store summary
        results.append(
            {
                "strategy": strategy_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

        print(f"\n=== {strategy_name} ===")
        print(report)
        print("Confusion matrix:")
        print(cm)

results_df = pd.DataFrame(results).sort_values(
    by=["f1", "recall", "precision"],
    ascending=False,
)

print("\n=== Final comparison ===")
print(results_df)

2026/03/28 18:13:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== none ===
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       400
           1       0.99      0.89      0.94       100

    accuracy                           0.98       500
   macro avg       0.98      0.94      0.96       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[399   1]
 [ 11  89]]


2026/03/28 18:13:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== class_weight_balanced ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       400
           1       0.92      0.93      0.93       100

    accuracy                           0.97       500
   macro avg       0.95      0.96      0.95       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix:
[[392   8]
 [  7  93]]


2026/03/28 18:13:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== random_oversample ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       400
           1       0.93      0.93      0.93       100

    accuracy                           0.97       500
   macro avg       0.96      0.96      0.96       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix:
[[393   7]
 [  7  93]]


2026/03/28 18:13:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== smote ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       400
           1       0.91      0.93      0.92       100

    accuracy                           0.97       500
   macro avg       0.95      0.95      0.95       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix:
[[391   9]
 [  7  93]]


2026/03/28 18:13:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== random_undersample ===
              precision    recall  f1-score   support

           0       0.98      0.97      0.98       400
           1       0.89      0.93      0.91       100

    accuracy                           0.96       500
   macro avg       0.94      0.95      0.94       500
weighted avg       0.96      0.96      0.96       500

Confusion matrix:
[[389  11]
 [  7  93]]

=== Final comparison ===
                strategy  accuracy  precision  recall        f1
0                   none     0.976   0.988889    0.89  0.936842
2      random_oversample     0.972   0.930000    0.93  0.930000
1  class_weight_balanced     0.970   0.920792    0.93  0.925373
3                  smote     0.968   0.911765    0.93  0.920792
4     random_undersample     0.964   0.894231    0.93  0.911765


 We see Logistic regression performs very well even in the no class imbalance management case, except in recall, where all the "fixed" cases performed the same, from this we understand that this is an easy problem, to study the best management method further we test another ML algorithm, Random forest

## Random Forest Baseline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

In [ ]:

numeric_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, numeric_columns),
        ("cat", categorical_transformer_tree, categorical_cols),
    ]
)

In [ ]:


rf_strategies = {
    "none": None,
    "class_weight_balanced": "balanced",
    "random_oversample": RandomOverSampler(random_state=42),
    "smote": SMOTE(random_state=42),
    "random_undersample": RandomUnderSampler(random_state=42),
}

rf_results = []

for strategy_name, strategy in rf_strategies.items():
    run_name = f"RF_{strategy_name}"

    with mlflow.start_run(run_name=run_name):
        if strategy_name == "class_weight_balanced":
            model = RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor_tree),   # use tree preprocessor
                    ("model", model),
                ]
            )

        elif strategy is None:
            model = RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            )

            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor_tree),
                    ("model", model),
                ]
            )

        else:
            model = RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            )

            pipeline = ImbPipeline(
                steps=[
                    ("preprocessor", preprocessor_tree),
                    ("sampler", strategy),
                    ("model", model),
                ]
            )

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_val)
        y_prob = None
        if hasattr(pipeline, "predict_proba"):
            y_prob = pipeline.predict_proba(X_val)[:, 1]

        accuracy = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)

        report = classification_report(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)

        mlflow.log_param("model", "RandomForestClassifier")
        mlflow.log_param("imbalance_strategy", strategy_name)
        mlflow.log_param("n_estimators", 200)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("n_numeric_cols", len(numeric_columns))
        mlflow.log_param("n_categorical_cols", len(categorical_cols))

        if strategy_name == "class_weight_balanced":
            mlflow.log_param("class_weight", "balanced")

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        report_path = f"rf_classification_report_{strategy_name}.txt"
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report)

        cm_path = f"rf_confusion_matrix_{strategy_name}.txt"
        with open(cm_path, "w", encoding="utf-8") as f:
            f.write(np.array2string(cm))

        mlflow.log_artifact(report_path)
        mlflow.log_artifact(cm_path)

        mlflow.sklearn.log_model(pipeline, name="model")

        rf_results.append(
            {
                "model": "RandomForestClassifier",
                "strategy": strategy_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

        print(f"\n=== RF {strategy_name} ===")
        print(report)
        print("Confusion matrix:")
        print(cm)

rf_results_df = pd.DataFrame(rf_results).sort_values(
    by=["f1", "recall", "precision"],
    ascending=False,
)

print("\n=== RF Final comparison ===")
print(rf_results_df)

2026/03/28 18:32:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RF none ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       400
           1       1.00      0.91      0.95       100

    accuracy                           0.98       500
   macro avg       0.99      0.96      0.97       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[400   0]
 [  9  91]]


2026/03/28 18:32:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RF class_weight_balanced ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       400
           1       1.00      0.91      0.95       100

    accuracy                           0.98       500
   macro avg       0.99      0.96      0.97       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[400   0]
 [  9  91]]


2026/03/28 18:32:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RF random_oversample ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       400
           1       0.99      0.93      0.96       100

    accuracy                           0.98       500
   macro avg       0.99      0.96      0.97       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[399   1]
 [  7  93]]


2026/03/28 18:32:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RF smote ===
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       400
           1       1.00      0.96      0.98       100

    accuracy                           0.99       500
   macro avg       1.00      0.98      0.99       500
weighted avg       0.99      0.99      0.99       500

Confusion matrix:
[[400   0]
 [  4  96]]


2026/03/28 18:32:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RF random_undersample ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       400
           1       0.96      0.95      0.95       100

    accuracy                           0.98       500
   macro avg       0.97      0.97      0.97       500
weighted avg       0.98      0.98      0.98       500

Confusion matrix:
[[396   4]
 [  5  95]]

=== RF Final comparison ===
                    model               strategy  accuracy  precision  recall  \
3  RandomForestClassifier                  smote     0.992   1.000000    0.96   
2  RandomForestClassifier      random_oversample     0.984   0.989362    0.93   
4  RandomForestClassifier     random_undersample     0.982   0.959596    0.95   
0  RandomForestClassifier                   none     0.982   1.000000    0.91   
1  RandomForestClassifier  class_weight_balanced     0.982   1.000000    0.91   

         f1  
3  0.979592  
2  0.958763  
4  0.954774  
0  0.952880  
1  0.952880  


we can see from this that model type is affecting performance more than class imbalance method, except at the none case where performance crashes, we will pause this part's exploration to focus on model type analysis and comparisn